In [ ]:
from pathlib import Path
from io import StringIO

import pandas as pd
import matplotlib.pyplot as plt

current_path = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (current_path, *current_path.parents)
        if (candidate / "data").is_dir() and (candidate / "src").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run this notebook from within the project directory.")
assets_dir = project_root / "assets"
chart_output_path = assets_dir / "basic_chart.png"
previous_notebook_path = project_root / "notebooks" / "03_datetime_missing_values.ipynb"

print("pandas_version:", pd.__version__)
print("project_root:", project_root)
print("assets_dir_exists:", assets_dir.exists())
print("previous_notebook_exists:", previous_notebook_path.exists())
print("chart_output_path:", chart_output_path)

In [ ]:
chart_sample_csv_text = """company_name,ticker,sector,report_date,revenue_usd_m_text,operating_income_usd_m_text,net_income_usd_m_text,unit,frequency,source_note
Apple Inc.,AAPL,consumer technology,2024-12-31,"1,000",300,250,USD million,annual_sample,class_sample
Microsoft Corporation,MSFT,software and cloud,2024-12-31,900,320,260,USD million,annual_sample,class_sample
NVIDIA Corporation,NVDA,semiconductor,2024-12-31,,280,220,USD million,annual_sample,class_sample
Apple Inc.,AAPL,consumer technology,2023-12-31,950,290,230,USD million,annual_sample,class_sample
Microsoft Corporation,MSFT,software and cloud,2023-12-31,880,,240,USD million,annual_sample,class_sample
NVIDIA Corporation,NVDA,semiconductor,2023-12-31,650,220,not available,USD million,annual_sample,class_sample
"""

raw_chart_df = pd.read_csv(StringIO(chart_sample_csv_text))

print("raw_chart_row_count:", len(raw_chart_df))
print("raw_chart_column_count:", len(raw_chart_df.columns))
raw_chart_df

In [ ]:
raw_chart_df["report_date"] = pd.to_datetime(raw_chart_df["report_date"])

numeric_source_columns = [
    "revenue_usd_m_text",
    "operating_income_usd_m_text",
    "net_income_usd_m_text",
]

numeric_clean_columns = [
    "revenue_usd_m",
    "operating_income_usd_m",
    "net_income_usd_m",
]

for source_column, clean_column in zip(numeric_source_columns, numeric_clean_columns):
    raw_chart_df[clean_column] = pd.to_numeric(
        raw_chart_df[source_column].astype("string").str.replace(",", "", regex=False),
        errors="coerce",
    )

chart_columns = [
    "company_name",
    "ticker",
    "sector",
    "report_date",
    "revenue_usd_m",
    "operating_income_usd_m",
    "net_income_usd_m",
    "unit",
    "frequency",
    "source_note",
]

chart_df = raw_chart_df[chart_columns].sort_values(["report_date", "ticker"]).copy()

print("date_dtype:", chart_df["report_date"].dtype)
print("chart_row_count:", len(chart_df))
print("chart_column_count:", len(chart_df.columns))
chart_df

In [ ]:
line_chart_df = chart_df.pivot(
    index="report_date",
    columns="ticker",
    values="revenue_usd_m",
)

print("line_chart_index_name:", line_chart_df.index.name)
print("line_chart_columns:", list(line_chart_df.columns))
line_chart_df

In [ ]:
if not assets_dir.exists():
    raise FileNotFoundError("The project assets directory is missing. Create it before saving a chart.")

ax = line_chart_df.plot(kind="line", marker="o")
ax.set_title("Sample revenue by report date")
ax.set_xlabel("Report date")
ax.set_ylabel("Revenue (USD millions)")
ax.legend(title="Ticker")

fig = ax.get_figure()
fig.tight_layout()
fig.savefig(chart_output_path, dpi=150)

print("saved_chart_path:", chart_output_path)
print("saved_chart_exists:", chart_output_path.exists())
plt.show()

In [ ]:
latest_report_date = chart_df["report_date"].max()
latest_operating_income_df = chart_df[chart_df["report_date"] == latest_report_date].sort_values("ticker")
latest_operating_income_order = (
    latest_operating_income_df
    .sort_values("operating_income_usd_m", ascending=False)["ticker"]
    .tolist()
)

print("latest_report_date:", latest_report_date.date())
print("latest_operating_income_row_count:", len(latest_operating_income_df))
print("latest_operating_income_order:", latest_operating_income_order)

ax = latest_operating_income_df.plot(
    kind="bar",
    x="ticker",
    y="operating_income_usd_m",
    legend=False,
)
ax.set_title(f"Sample operating income on {latest_report_date:%Y-%m-%d}")
ax.set_xlabel("Ticker")
ax.set_ylabel("Operating income (USD millions)")
plt.tight_layout()
plt.show()

latest_operating_income_df[["ticker", "operating_income_usd_m", "unit", "source_note"]]

In [ ]:
revenue_change_by_ticker = (
    line_chart_df.loc[pd.Timestamp("2024-12-31")]
    - line_chart_df.loc[pd.Timestamp("2023-12-31")]
)

interpretation_sentences = []

for ticker, change_value in revenue_change_by_ticker.items():
    period = f"From 2023-12-31 to 2024-12-31, {ticker}'s sample revenue"
    if pd.isna(change_value):
        sentence = f"{period} cannot be compared because a value is missing."
    elif change_value > 0:
        sentence = f"{period} increased by USD {change_value:.0f} million."
    elif change_value < 0:
        sentence = f"{period} decreased by USD {abs(change_value):.0f} million."
    else:
        sentence = f"{period} was unchanged."
    interpretation_sentences.append(sentence)

for sentence in interpretation_sentences:
    print(sentence)
